In [42]:
import pandas as pd
import sqlalchemy
from sqlalchemy import create_engine

In [43]:
db_config = {
    'user': 'practicum_student',
    'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-final-project-db'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode': 'require'})

In [44]:
# ¿Qué tablas hay?
query = """
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'public'
"""

pd.read_sql(query, engine)

,table_name
0,ratings
1,advertisment_costs
2,authors
3,orders
4,reviews
5,visits
6,books
7,users
8,publishers


In [45]:
book = pd.read_sql("SELECT * FROM books", engine)
book.columns

Index(['book_id', 'author_id', 'title', 'num_pages', 'publication_date',
       'publisher_id'],
      dtype='object')

In [46]:
query = """
SELECT COUNT(*)
FROM books
WHERE publication_date > '2000-01-01';
"""

pd.read_sql(query, engine)# ¿Cuántos libros se publicaron despues del primero de enero del 2000?


,count
0,819


Después del primero de enero del año 2000, se publicaron 819 libros.

In [47]:
# ¿Cuántas reseñas hay  por libro y cual es la puntuación promedio para cada uno?
# Subconsulta para número de reviews
query = """
SELECT 
    book_id,
    COUNT(review_id) AS total_reviews
FROM reviews
GROUP BY book_id
"""
pd.read_sql(query, engine)


,book_id,total_reviews
0,652,2
1,273,2
2,51,5
3,951,2
4,839,4
...,...,...
989,64,4
990,55,2
991,148,3
992,790,2


In [48]:
# Subconsulta para ratings promedio
query = """
SELECT 
    book_id,
    AVG(rating) AS avg_rating
FROM ratings
GROUP BY book_id
"""

pd.read_sql(query, engine)

,book_id,avg_rating
0,652,4.500000
1,273,4.500000
2,51,4.250000
3,951,4.000000
4,839,4.285714
...,...,...
995,64,4.230769
996,55,5.000000
997,148,3.428571
998,790,3.500000


In [49]:
# Consulta uniendo las tablas books y publisher
query = """
SELECT 
    p.publisher,
    COUNT(b.book_id) AS total_books
FROM books b
JOIN publishers p 
    ON b.publisher_id = p.publisher_id
WHERE b.num_pages > 50
GROUP BY p.publisher
ORDER BY total_books DESC
LIMIT 5;
"""

pd.read_sql(query, engine)

,publisher,total_books
0,Penguin Books,42
1,Vintage,31
2,Grand Central Publishing,25
3,Penguin Classics,24
4,Bantam,19


La editorial con mayor número de úblicaciones fue Penguin Books, con un total de 42.

In [50]:
query = """
SELECT 
    a.author,
    AVG(rt.avg_rating) AS author_avg_rating
FROM (
    SELECT 
        book_id,
        COUNT(rating_id) AS total_ratings,
        AVG(rating) AS avg_rating
    FROM ratings
    GROUP BY book_id
    HAVING COUNT(rating_id) >= 50
) rt
JOIN books b 
    ON rt.book_id = b.book_id
JOIN authors a 
    ON b.author_id = a.author_id
GROUP BY a.author
ORDER BY author_avg_rating DESC
LIMIT 1;
"""

pd.read_sql(query, engine)

,author,author_avg_rating
0,J.K. Rowling/Mary GrandPré,4.283844


In [51]:
query = """
WITH active_users AS (
    SELECT username
    FROM ratings
    GROUP BY username
    HAVING COUNT(rating_id) > 50
)

SELECT AVG(review_count) AS avg_reviews
FROM (
    SELECT 
        au.username,
        COUNT(rv.review_id) AS review_count
    FROM active_users au
    LEFT JOIN reviews rv
        ON au.username = rv.username
    GROUP BY au.username
) user_reviews;
"""

pd.read_sql(query, engine)

,avg_reviews
0,24.333333
